In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Transpiler-Optimierungsstufe festlegen

<details>
<summary><b>Package versions</b></summary>

Der Code auf dieser Seite wurde mit den folgenden Anforderungen entwickelt.
Wir empfehlen, diese Versionen oder neuere zu verwenden.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Echte Quantengeräte unterliegen Rauschen und Gate-Fehlern, weshalb die Optimierung von Schaltkreisen zur Reduzierung ihrer Tiefe und Gate-Anzahl die Ergebnisse, die durch das Ausführen dieser Schaltkreise erzielt werden, erheblich verbessern kann.
Die Funktion [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) hat ein erforderliches Positionsargument, `optimization_level`, das steuert, wie viel Aufwand der Transpiler für die Optimierung von Schaltkreisen aufwendet. Dieses Argument kann eine ganze Zahl sein, die einen der Werte 0, 1, 2 oder 3 annimmt.
Höhere Optimierungsstufen erzeugen stärker optimierte Schaltkreise auf Kosten längerer Kompilierungszeiten.
Die folgende Tabelle erläutert die Optimierungen, die mit jeder Einstellung durchgeführt werden.

<table>
  <thead>
    <tr>
      <th>Optimierungsstufe</th>
      <th>Beschreibung</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>0</td>
      <td>
        Keine Optimierung: typischerweise für Hardware-Charakterisierung verwendet
        - Grundlegende Übersetzung
        - Layout/Routing: `TrivialLayout`, wobei dieselben physikalischen Qubit-Nummern wie virtuelle ausgewählt und SWAPs eingefügt werden, um es zum Laufen zu bringen (unter Verwendung von `SabreSwap`)
      </td>
    </tr>
    <tr>
      <td>1</td>
      <td>
        Leichte Optimierung:
        -   Layout/Routing: Das Layout wird zunächst mit `TrivialLayout` versucht. Wenn zusätzliche SWAPs erforderlich sind, wird ein Layout mit einer minimalen Anzahl von SWAPs mithilfe von `SabreSwap` gefunden, dann verwendet es `VF2LayoutPostLayout`, um die besten Qubits im Graphen auszuwählen.
        -   `InverseCancellation`
        -   1Q-Gate-Optimierung
      </td>
    </tr>
    <tr>
      <td>2</td>
      <td>
        Mittlere Optimierung:
          - Layout/Routing: Optimierungsstufe 1 (ohne trivial) + heuristisch optimiert mit größerer
        Suchtiefe und Versuchen der Optimierungsfunktion. Da `TrivialLayout` nicht verwendet wird, wird kein Versuch unternommen, dieselben physikalischen und virtuellen Qubit-Nummern zu verwenden.
        -   `CommutativeCancellation`
      </td>
    </tr>
    <tr>
      <td>3</td>
      <td>
        Hohe Optimierung:
        - Optimierungsstufe 2 + heuristisch weiter auf Layout/Routing optimiert mit größerem Aufwand/Versuchen
        - Resynthese von Zwei-Qubit-Blöcken unter Verwendung der [Cartan KAK-Zerlegung](https://arxiv.org/abs/quant-ph/0507171).
        - Unitaritätsbrechende Passes:
          * `OptimizeSwapBeforeMeasure`: Verschiebt die Messungen, um SWAPs zu vermeiden
          * `RemoveDiagonalGatesBeforeMeasure`: Entfernt Gates vor Messungen, die die Messungen nicht beeinflussen würden
      </td>
    </tr>
  </tbody>
</table>

## Optimierungsstufe in der Praxis
Da Zwei-Qubit-Gates typischerweise die bedeutendste Fehlerquelle sind, können wir die „Hardware-Effizienz" der Transpilierung annäherungsweise quantifizieren, indem wir die Anzahl der Zwei-Qubit-Gates im resultierenden Schaltkreis zählen.
Hier probieren wir die verschiedenen Optimierungsstufen auf einem Eingangsschaltkreis aus, der aus einer zufälligen unitären Matrix gefolgt von einem SWAP-Gate besteht.

In [1]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import UnitaryGate
from qiskit.quantum_info import Operator, random_unitary

UU = random_unitary(4, seed=12345)
rand_U = UnitaryGate(UU)

qc = QuantumCircuit(2)
qc.append(rand_U, range(2))
qc.swap(0, 1)
qc.draw("mpl", style="iqp")

<Image src="../docs/images/guides/set-optimization/extracted-outputs/81173ebc-8359-48a6-b585-0477907b3b93-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/81173ebc-8359-48a6-b585-0477907b3b93-0.svg)

Wir verwenden das `FakeSherbrooke`-Mock-Backend in unseren Beispielen. Zuerst transpilieren wir mit Optimierungsstufe 0.

In [2]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()

pass_manager = generate_preset_pass_manager(
    optimization_level=0, backend=backend, seed_transpiler=12345
)
qc_t1_exact = pass_manager.run(qc)
qc_t1_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/40cdd173-b437-48b1-8928-741e8411342e-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/40cdd173-b437-48b1-8928-741e8411342e-0.svg)

Der transpilierte Schaltkreis hat sechs Zwei-Qubit-ECR-Gates.

Wiederholen für Optimierungsstufe 1:

In [3]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()

pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend, seed_transpiler=12345
)
qc_t1_exact = pass_manager.run(qc)
qc_t1_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/2dab5def-a017-42e9-92d6-e043ac4065b2-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/2dab5def-a017-42e9-92d6-e043ac4065b2-0.svg)

Der transpilierte Schaltkreis hat immer noch sechs ECR-Gates, aber die Anzahl der Einzel-Qubit-Gates hat sich verringert.

Wiederholen für Optimierungsstufe 2:

In [4]:
pass_manager = generate_preset_pass_manager(
    optimization_level=2, backend=backend, seed_transpiler=12345
)
qc_t2_exact = pass_manager.run(qc)
qc_t2_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/77d76048-b1e8-4225-b35f-80dc9d458e8d-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/77d76048-b1e8-4225-b35f-80dc9d458e8d-0.svg)

Das ergibt dieselben Ergebnisse wie Optimierungsstufe 1. Beachte, dass eine Erhöhung der Optimierungsstufe nicht immer einen Unterschied macht.

Nochmals wiederholen, mit Optimierungsstufe 3:

In [5]:
pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend, seed_transpiler=12345
)
qc_t3_exact = pass_manager.run(qc)
qc_t3_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/4109d0e2-df37-4850-8409-6b860c48595c-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/4109d0e2-df37-4850-8409-6b860c48595c-0.svg)

Jetzt gibt es nur noch drei ECR-Gates. Dieses Ergebnis erhalten wir, weil Qiskit bei Optimierungsstufe 3 versucht, Zwei-Qubit-Gate-Blöcke neu zu synthetisieren, und jedes Zwei-Qubit-Gate mit höchstens drei ECR-Gates implementiert werden kann. Wir können sogar noch weniger ECR-Gates erhalten, wenn wir `approximation_degree` auf einen Wert kleiner als 1 setzen, sodass der Transpiler Näherungen vornehmen darf, die möglicherweise einen gewissen Fehler in der Gate-Zerlegung einführen (siehe [Häufig verwendete Parameter für die Transpilierung](common-parameters#approximation-degree)):

In [6]:
pass_manager = generate_preset_pass_manager(
    optimization_level=3,
    approximation_degree=0.99,
    backend=backend,
    seed_transpiler=12345,
)
qc_t3_approx = pass_manager.run(qc)
qc_t3_approx.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/bf239116-b8bb-42aa-a27a-89206d9e108a-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/bf239116-b8bb-42aa-a27a-89206d9e108a-0.svg)

Dieser Schaltkreis hat nur zwei ECR-Gates, ist aber ein Näherungsschaltkreis. Um zu verstehen, wie sich seine Wirkung vom exakten Schaltkreis unterscheidet, können wir die Fidelität zwischen dem unitären Operator, den dieser Schaltkreis implementiert, und dem exakten Unitären berechnen. Vor der Berechnung reduzieren wir zunächst den transpilierten Schaltkreis, der 127 Qubits enthält, auf einen Schaltkreis, der nur die aktiven Qubits enthält, von denen es zwei gibt.

In [7]:
import numpy as np


def trace_to_fidelity_2q(trace: float) -> float:
    return (4.0 + trace * trace.conjugate()) / 20.0


# Reduce circuits down to 2 qubits so they are easy to simulate
qc_t3_exact_small = QuantumCircuit.from_instructions(qc_t3_exact)
qc_t3_approx_small = QuantumCircuit.from_instructions(qc_t3_approx)

# Compute the fidelity
exact_fid = trace_to_fidelity_2q(
    np.trace(np.dot(Operator(qc_t3_exact_small).adjoint().data, UU))
)
approx_fid = trace_to_fidelity_2q(
    np.trace(np.dot(Operator(qc_t3_approx_small).adjoint().data, UU))
)
print(
    f"Synthesis fidelity\nExact: {exact_fid:.3f}\nApproximate: {approx_fid:.3f}"
)

Synthesis fidelity
Exact: 1.000+0.000j
Approximate: 0.992+0.000j


Adjusting the optimization level can change other aspects of the circuit too, not just the number of ECR gates. For examples of how setting optimization level changes the layout, see [Representing quantum computers](./represent-quantum-computers).

## Next steps

<Admonition type="tip" title="Recommendations">
    - To learn more about the `generate_preset_passmanager` function, start with the [Transpilation default settings and configuration options](defaults-and-configuration-options) topic.
    - Continue learning about transpilation with the [Transpiler stages](transpiler-stages) topic.
    - Try the [Compare transpiler settings](/docs/guides/circuit-transpilation-settings) guide.
    - Try the [Build repetition codes](/docs/tutorials/repetition-codes) tutorial.
    - See the [Transpile API documentation.](/docs/api/qiskit/transpiler)
</Admonition>